# IFC Export

**Stage 8**  - Build the final IFC4 BIM model from room wall clouds.

This notebook:
1. Segments each room's wall cloud into planar walls
2. Computes wall geometry (start/end points, height, thickness)
3. Merges duplicate wall faces and deduplicates shared walls between rooms
4. Detects doors on merged wall surfaces
5. Snaps wall angles to Manhattan directions and endpoints to junctions
6. Writes the result as a canonical JSON and an IFC4 file

**Input:** Room wall clouds (`.ply`) from any upstream wall-assignment stage.

**Output:** `building.json` (canonical wall/door/room data) + `model.ifc` (IFC4 file).

**Requires:** `ifcopenshell` (`pip install ifcopenshell`) for the IFC4 writer.

In [ ]:
import sys, os, glob, json, logging

def _find_root():
    d = os.path.abspath('')
    while True:
        if os.path.isfile(os.path.join(d, 'pyproject.toml')):
            return d
        p = os.path.dirname(d)
        if p == d:
            return os.path.abspath('')
        d = p

PROJECT_ROOT = _find_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.path = [p for p in sys.path if not p.endswith('/src') and not p.endswith('\\src')]
for mod in list(sys.modules):
    if mod == 'scan2bim' or mod.startswith('scan2bim.'):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format='%(message)s')
print('Project root:', PROJECT_ROOT)

In [ ]:
import scan2bim
from scan2bim import artifacts as A

CFG = scan2bim.load_config(start=PROJECT_ROOT)
print(f'IFC project name: {CFG.ifc_project_name}')
print(f'Default wall thickness: {CFG.ifc_default_thickness}m')
print(f'Min wall length: {CFG.ifc_min_wall_length_m}m')

## Select upstream wall stage

In [ ]:
# Choose one:
WALL_STAGE = A.STAGE3          # geometric method
# WALL_STAGE = A.STAGE_SAM_WALLS  # pure-SAM method
# WALL_STAGE = A.STAGE5          # geometric + SAM method

wall_dir = A.load_stage_dir(CFG.out_root, WALL_STAGE)
room_clouds = sorted(glob.glob(os.path.join(wall_dir, 'room_*_walls.ply')))
print(f'Found {len(room_clouds)} room wall clouds in {WALL_STAGE}')
for p in room_clouds:
    print(f'  {os.path.basename(p)}')

## Build the canonical building JSON

This step segments walls, computes geometry, merges/deduplicates, detects doors,
and produces a structured JSON with walls, doors, and room boundaries.

In [ ]:
from scan2bim.ifc_export import build_building_json

out_dir = A.ensure_dir(A.stage_dir(CFG.out_root, A.STAGE_IFC))
building = build_building_json(room_clouds, CFG, out_dir=out_dir)

# Save JSON
json_path = os.path.join(out_dir, A.BUILDING_JSON)
# Strip debug data for the saved version
save_data = {k: v for k, v in building.items() if k != '_debug'}
with open(json_path, 'w') as f:
    json.dump(save_data, f, indent=2)
print(f'\nSaved: {json_path}')
print(f'  Walls: {len(building["walls"])}')
print(f'  Doors: {len(building["doors"])}')
print(f'  Rooms: {len(building["rooms"])}')

## Visualize the floor plan

In [ ]:
from scan2bim.viz import show_floor_plan

show_floor_plan(building)

## Write IFC4 file

Requires `ifcopenshell`. If not installed, skip this cell  - the building JSON above
is the canonical output and can be converted to IFC later.

In [ ]:
from scan2bim.ifc_export import build_ifc

ifc_path = os.path.join(out_dir, A.IFC_MODEL)
try:
    build_ifc(building, out_path=ifc_path, cfg=CFG)
    print(f'\nIFC model written to: {ifc_path}')
except ImportError:
    print('ifcopenshell not installed  - skipping IFC export.')
    print('Install with: pip install ifcopenshell')
    print(f'The building JSON at {json_path} is the canonical output.')

In [ ]:
A.save_config(os.path.join(out_dir, A.CONFIG_JSON), CFG)
z = A.package_stage(CFG.out_root, A.STAGE_IFC)
print(f'Packaged: {z}')